In [0]:

dbutils.widgets.text("catalogo", "catalog_dev")
dbutils.widgets.text("storageName", "adlsmartdata08100")

catalogo = dbutils.widgets.get("catalogo")
storageName = dbutils.widgets.get("storageName")

ruta_raw = (
    f"abfss://raw@{storageName}.dfs.core.windows.net/"
)

archivos_bronze = [
    ("olist_orders_dataset.csv", "orders", 99441),
    ("olist_order_items_dataset.csv", "order_items", 112650),
    ("olist_customers_dataset.csv", "customers", 99441),
    ("olist_products_dataset.csv", "products", 32951),
    ("olist_sellers_dataset.csv", "sellers", 3095),
    ("olist_order_payments_dataset.csv", "order_payments", 103886),
    ("olist_order_reviews_dataset.csv", "order_reviews", 99224),
    ("olist_geolocation_dataset.csv", "geolocation", 1000163),
    ("product_category_name_translation.csv", "category_translation", 71)
]

print("Catálogo:", catalogo)
print("Origen:", ruta_raw)
print("Archivos configurados:", len(archivos_bronze))


In [0]:

df_lecturas = {}

for archivo, tabla, esperado in archivos_bronze:

    ruta = ruta_raw + archivo
    destino = f"{catalogo}.bronze.{tabla}"

    esquema = spark.table(destino).schema

    df = (
        spark.read
        .format("csv")
        .option("header", "true")
        .option("sep", ",")
        .option("quote", '"')
        .option("escape", '"')
        .option("multiLine", str(tabla == "order_reviews").lower())
        .option("mode", "FAILFAST")
        .option("enforceSchema", "false")
        .schema(esquema)
        .load(ruta)
    )

    total = df.count()

    if total != esperado:
        raise ValueError(
            f"{archivo}: se esperaban {esperado} "
            f"registros, pero se encontraron {total}"
        )

    df_lecturas[tabla] = df

    print(f"OK | {tabla}: {total:,} registros")

print("\nValidación de los 9 CSV completada.")


In [0]:

for archivo, tabla, esperado in archivos_bronze:

    destino = f"{catalogo}.bronze.{tabla}"

    df = df_lecturas[tabla]

    df.write.mode("overwrite").insertInto(destino)

    total = spark.table(destino).count()

    if total != esperado:
        raise ValueError(
            f"Error en {destino}: "
            f"esperados={esperado}, cargados={total}"
        )

    print(f"OK | {destino}: {total:,} registros")

print("\nBRONZE COMPLETADO: 9 tablas cargadas correctamente.")
